In [ ]:
pip install beautifulsoup4 requests lxml pandas tqdm

In [ ]:
import os
import re
import time
import random
import threading
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from requests.adapters import HTTPAdapter, Retry
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

BASE_URL = "https://merolagani.com/Floorsheet.aspx"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://merolagani.com",
    "Content-Type": "application/x-www-form-urlencoded",
    "Referer": BASE_URL,
}

DATA_DIR = "data/floorsheet"
os.makedirs(DATA_DIR, exist_ok=True)

# How many concurrent HTTP requests we allow in flight at once. Keep this
# moderate to stay reliable against the server rather than maximally aggressive.
MAX_WORKERS = 10
PAGE_SIZE = 500  # rows per page, fixed by the site


In [ ]:
def make_session():
    """A requests.Session with retry/backoff on transient failures."""
    s = requests.Session()
    s.headers.update(HEADERS)
    retries = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    )
    adapter = HTTPAdapter(max_retries=retries, pool_maxsize=MAX_WORKERS * 2)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


def extract_form_fields(html: str) -> dict:
    """Pull every <input name=...> inside the aspnetForm — WebForms postbacks
    require the full form state (viewstate, event validation, filters, pager
    hidden fields) to be echoed back exactly as a browser would."""
    soup = BeautifulSoup(html, "lxml")
    form = soup.find("form", id="aspnetForm")
    fields = {
        inp.get("name"): inp.get("value", "")
        for inp in form.find_all("input")
        if inp.get("name")
    }
    return fields


RECORD_RE = re.compile(
    r"Showing ([\d,]+) - ([\d,]+) of ([\d,]+) records\. \[Total pages: (\d+)\]"
)


def parse_record_summary(html: str):
    m = RECORD_RE.search(html)
    if not m:
        return None
    _, _, total_records, total_pages = m.groups()
    return int(total_records.replace(",", "")), int(total_pages)


def parse_floorsheet_rows(html: str, trade_date: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    table = soup.select_one("#ctl00_ContentPlaceHolder1_divData table.sortable")
    if table is None:
        return []
    rows = []
    for tr in table.select("tbody tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) != 8:
            continue
        _, txn_no, symbol, buyer, seller, qty, rate, amount = cells
        rows.append({
            "date": trade_date,
            "transaction_no": txn_no,
            "symbol": symbol,
            "buyer": buyer,
            "seller": seller,
            "quantity": int(qty.replace(",", "") or 0),
            "rate": float(rate.replace(",", "") or 0),
            "amount": float(amount.replace(",", "") or 0),
        })
    return rows


In [ ]:
def search_date(session: requests.Session, mmddyyyy: str):
    """Load the page fresh, then submit the date filter via the same postback
    the 'Search' link fires client-side (__EVENTTARGET=lbtnSearchFloorsheet).
    Returns (fields_after_search, total_records, total_pages, page1_rows).
    fields_after_search carries the viewstate/eventvalidation snapshot that
    resulted from applying this date filter — every subsequent page request
    for this date forks off of it.
    """
    r0 = session.get(BASE_URL, timeout=30)
    r0.raise_for_status()
    fields = extract_form_fields(r0.text)

    payload = dict(fields)
    payload["ctl00$ContentPlaceHolder1$txtFloorsheetDateFilter"] = mmddyyyy
    payload["__EVENTTARGET"] = "ctl00$ContentPlaceHolder1$lbtnSearchFloorsheet"
    payload["__EVENTARGUMENT"] = ""
    # Submit-button fields must NOT be present alongside a __EVENTTARGET postback,
    # or the server errors out (it saw two conflicting triggers).
    payload.pop("ctl00$ContentPlaceHolder1$PagerControl1$btnPaging", None)
    payload.pop("ctl00$ContentPlaceHolder1$PagerControl2$btnPaging", None)

    r1 = session.post(BASE_URL, data=payload, timeout=30)
    r1.raise_for_status()

    summary = parse_record_summary(r1.text)
    if summary is None:
        # No trades that day (holiday/weekend) or the date filter produced no rows.
        return None
    total_records, total_pages = summary
    fields_after_search = extract_form_fields(r1.text)
    page1_rows = parse_floorsheet_rows(r1.text, iso_date_from_mmddyyyy(mmddyyyy))
    return fields_after_search, total_records, total_pages, page1_rows


def fetch_page(session: requests.Session, fields_after_search: dict, mmddyyyy: str, page_num: int, attempts: int = 3):
    """Fetch a single page by forking off the post-search viewstate snapshot.
    Verified empirically: the server does not require pages to be requested
    in sequence — many pages can be requested concurrently from the same
    base viewstate, which is what makes parallel extraction possible here.
    """
    payload = dict(fields_after_search)
    payload.pop("__EVENTTARGET", None)
    payload.pop("__EVENTARGUMENT", None)
    payload.pop("ctl00$ContentPlaceHolder1$PagerControl2$btnPaging", None)
    payload["ctl00$ContentPlaceHolder1$PagerControl1$hdnCurrentPage"] = str(page_num)

    for attempt in range(attempts):
        try:
            r = session.post(BASE_URL, data=payload, timeout=30)
            r.raise_for_status()
            rows = parse_floorsheet_rows(r.text, iso_date_from_mmddyyyy(mmddyyyy))
            if rows:
                return rows
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
        time.sleep(0.5 * (attempt + 1))
    return []


def iso_date_from_mmddyyyy(mmddyyyy: str) -> str:
    m, d, y = mmddyyyy.split("/")
    return f"{y}-{m}-{d}"


def fetch_floorsheet_for_date(mmddyyyy: str, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    """The underlying grid paginates with OFFSET/FETCH but no stable tiebreaker
    in its ORDER BY, so a row can occasionally land on a different page than
    expected between two requests — a boundary row appears twice (or briefly
    seems to vanish) rather than data being genuinely lost. We fetch every
    page and de-duplicate on transaction_no (the real unique key) instead of
    trusting exact per-page row counts.
    """
    session = make_session()
    result = search_date(session, mmddyyyy)
    if result is None:
        return pd.DataFrame()
    fields_after_search, total_records, total_pages, page1_rows = result

    all_rows = list(page1_rows)
    remaining_pages = list(range(2, total_pages + 1))

    if remaining_pages:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {
                pool.submit(fetch_page, session, fields_after_search, mmddyyyy, p): p
                for p in remaining_pages
            }
            for fut in as_completed(futures):
                all_rows.extend(fut.result())

    df = pd.DataFrame(all_rows).drop_duplicates(subset="transaction_no")
    shortfall = total_records - len(df)
    if abs(shortfall) > max(5, total_records * 0.001):
        raise RuntimeError(
            f"{mmddyyyy}: got {len(df)} unique rows, expected ~{total_records} (off by {shortfall})"
        )
    return df.reset_index(drop=True)


In [ ]:
# Quick smoke test on a single known trading date before running the full backfill.
_test_df = fetch_floorsheet_for_date("06/30/2026")
print(len(_test_df), "rows")
_test_df.head()


In [ ]:
def trading_dates(start: date, end: date):
    """NEPSE trades Sunday-Thursday; Friday(4) and Saturday(5) are closed."""
    d = start
    while d <= end:
        if d.weekday() not in (4, 5):
            yield d
        d += timedelta(days=1)


def date_csv_path(d: date) -> str:
    return os.path.join(DATA_DIR, f"{d.isoformat()}.csv")


def backfill_floorsheet(start: date, end: date, max_workers: int = MAX_WORKERS, date_workers: int = 3):
    """Fetch and cache one CSV per trading day. Already-downloaded dates are
    skipped, so this is safe to re-run/resume. Multiple dates are processed
    concurrently on top of the per-date page-level parallelism above.
    """
    dates = [d for d in trading_dates(start, end) if not os.path.exists(date_csv_path(d))]
    if not dates:
        print("Nothing to do — all dates already cached.")
        return

    def _do(d):
        mmddyyyy = d.strftime("%m/%d/%Y")
        try:
            df = fetch_floorsheet_for_date(mmddyyyy, max_workers=max_workers)
        except Exception as e:
            return d, None, e
        return d, df, None

    with ThreadPoolExecutor(max_workers=date_workers) as pool:
        futures = [pool.submit(_do, d) for d in dates]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="dates"):
            d, df, err = fut.result()
            if err is not None:
                print(f"FAILED {d}: {err}")
                continue
            if df.empty:
                # No trading that day (market holiday) — write an empty marker
                # so we don't keep retrying it on resume.
                pd.DataFrame(columns=["date", "transaction_no", "symbol", "buyer", "seller", "quantity", "rate", "amount"]).to_csv(date_csv_path(d), index=False)
            else:
                df.to_csv(date_csv_path(d), index=False)


In [ ]:
# Example: backfill a date range. Adjust the start date to how far back you want
# history — spot-checks show merolagani's floorsheet archive has real data back
# to at least mid-2016; for dates it doesn't have, the search just returns zero
# rows and backfill_floorsheet() writes an empty marker CSV so it isn't retried.
# backfill_floorsheet(date(2016, 1, 1), date.today() - timedelta(days=1))


In [ ]:
def build_ohlcv(floorsheet_dir: str = DATA_DIR) -> pd.DataFrame:
    """Roll up cached per-day floorsheet CSVs into a daily OHLCV table per symbol.

    The floorsheet has no per-trade timestamp, but transaction_no is a
    strictly increasing counter within a trading day, so it stands in for
    trade sequence: the lowest transaction_no of the day is the first trade
    (Open) and the highest is the last (Close).
    """
    files = sorted(f for f in os.listdir(floorsheet_dir) if f.endswith(".csv"))
    frames = []
    for f in tqdm(files, desc="loading"):
        df = pd.read_csv(os.path.join(floorsheet_dir, f), dtype={"transaction_no": str})
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()

    fs = pd.concat(frames, ignore_index=True)
    fs = fs.sort_values(["date", "symbol", "transaction_no"])

    grouped = fs.groupby(["date", "symbol"])
    ohlcv = grouped.agg(
        open=("rate", "first"),
        high=("rate", "max"),
        low=("rate", "min"),
        close=("rate", "last"),
        volume=("quantity", "sum"),
        turnover=("amount", "sum"),
        trades=("transaction_no", "count"),
    ).reset_index()

    return ohlcv.sort_values(["symbol", "date"]).reset_index(drop=True)


# ohlcv = build_ohlcv()
# ohlcv.to_csv("nepse_ohlcv.csv", index=False)
# ohlcv.head()
